# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [5]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_contents , fetch_website_links
from openai import OpenAI


In [7]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [8]:
links = fetch_website_links("https://shivanandkumar.in")
links

['#top',
 '#impact',
 '#projects',
 '#profile',
 '#contact',
 'https://www.iitp.ac.in/',
 '#impact',
 '#projects',
 '/projects/data-platform-reliability-agent/',
 'https://github.com/omni-shiva/shiva-applied-agentic/tree/main/projects/data-platform-reliability-agent',
 '/projects/synthetic-data-print-recommendation-agent/',
 'https://github.com/omni-shiva/shiva-applied-agentic/tree/main/projects/synthetic-data-print-recommendation-agent',
 '/projects/constraint-aware-coding-agent-evals/',
 'https://github.com/omni-shiva/constraint-aware-coding-agent-evals',
 '#projects',
 'mailto:kumarshivanand7@gmail.com',
 'https://linkedin.com/in/shivachauhan',
 'https://github.com/omni-shiva',
 'https://www.hackerrank.com/profile/krshivan']

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [10]:
link_system_prompt="""
You are provided with a list of links found on a webpage , You areable to decide which of the 
link would be most relevant to include in a broucher about the company,
such as links to an about page, or a company name , or carrer / jobs pages .
You should respon in JSON as in this example:
{ "links":[
        {"type":"about page","url":"https://full.url/goes/here/about"},
        {"type":"careers page","url":"https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt=f"""
    Here is the list of link on the website {url} - 
    please decide which
    
    
    
    """

In [11]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [12]:
print(get_links_user_prompt("https://shivanandkumar.in"))


Here is the list of links on the website https://shivanandkumar.in -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#top
#impact
#projects
#profile
#contact
https://www.iitp.ac.in/
#impact
#projects
/projects/data-platform-reliability-agent/
https://github.com/omni-shiva/shiva-applied-agentic/tree/main/projects/data-platform-reliability-agent
/projects/synthetic-data-print-recommendation-agent/
https://github.com/omni-shiva/shiva-applied-agentic/tree/main/projects/synthetic-data-print-recommendation-agent
/projects/constraint-aware-coding-agent-evals/
https://github.com/omni-shiva/constraint-aware-coding-agent-evals
#projects
mailto:kumarshivanand7@gmail.com
https://linkedin.com/in/shivachauhan
https://github.com/omni-shiva
https://www.hackerrank.com/profile/krshivan


In [36]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[{"role":"system","content":link_system_prompt},
                {"role":"user","content": get_links_user_prompt(url)}],
        response_format={"type":"json_object"}
     ) #doing llm select and enforcing format .. while inference
    #when we say pick the next token it means that pick the highest next token.)
    result = response.choices[0].message.content
    links = json.loads(result)
    return links



In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [38]:
select_relevant_links("https://shivanandkumar.in")

{'links': [{'type': 'portfolio page', 'url': 'https://github.com/omni-shiva'},
  {'type': 'project repository',
   'url': 'https://github.com/omni-shiva/shiva-applied-agentic/tree/main/projects/data-platform-reliability-agent'},
  {'type': 'project repository',
   'url': 'https://github.com/omni-shiva/shiva-applied-agentic/tree/main/projects/synthetic-data-print-recommendation-agent'},
  {'type': 'project repository',
   'url': 'https://github.com/omni-shiva/constraint-aware-coding-agent-evals'},
  {'type': 'academic affiliation', 'url': 'https://www.iitp.ac.in/'}]}

In [39]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [42]:
select_relevant_links("https://shivanandkumar.in")

Selecting relevant links for https://shivanandkumar.in by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/in/shivachauhan'},
  {'type': 'GitHub profile', 'url': 'https://github.com/omni-shiva'},
  {'type': 'GitHub project page',
   'url': 'https://github.com/omni-shiva/shiva-applied-agentic/tree/main/projects/data-platform-reliability-agent'},
  {'type': 'GitHub project page',
   'url': 'https://github.com/omni-shiva/shiva-applied-agentic/tree/main/projects/synthetic-data-print-recommendation-agent'},
  {'type': 'GitHub repository',
   'url': 'https://github.com/omni-shiva/constraint-aware-coding-agent-evals'}]}

In [49]:
select_relevant_links("https://shivanandkumar.in")

Selecting relevant links for https://shivanandkumar.in by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'GitHub organization',
   'url': 'https://github.com/omni-shiva'},
  {'type': 'GitHub project - data platform reliability agent',
   'url': 'https://github.com/omni-shiva/shiva-applied-agentic/tree/main/projects/data-platform-reliability-agent'},
  {'type': 'GitHub project - synthetic data print-recommendation agent',
   'url': 'https://github.com/omni-shiva/shiva-applied-agentic/tree/main/projects/synthetic-data-print-recommendation-agent'},
  {'type': 'GitHub project - constraint-aware coding agent evals',
   'url': 'https://github.com/omni-shiva/constraint-aware-coding-agent-evals'},
  {'type': 'Affiliations / Institution', 'url': 'https://www.iitp.ac.in/'},
  {'type': 'LinkedIn profile', 'url': 'https://linkedin.com/in/shivachauhan'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [50]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [51]:
print(fetch_page_and_all_relevant_links("https://shivanandkumar.in"))

Selecting relevant links for https://shivanandkumar.in by calling gpt-5-nano
Found 7 relevant links
## Landing Page:

Shivanand Kumar | Data & Applied AI Engineer

Shivanand Kumar
Impact
Projects
Experience
Contact
01
Context
Data · contracts · signals
02
Retrieval
Grounded evidence
AGENTIC
Workflow
04
Evaluation
Quality · safety · traces
05
Human approval
Controlled action
Bengaluru, India
IIT Patna
M.Tech · AI & Data Science Engineering
Data & Applied AI Engineer
Generative AI · Agentic AI · Databricks · Spark
I build reliable data and AI systems—from Databricks and Spark platforms to GenAI automation, agentic workflows, retrieval and evaluation.
See production impact
Inspect public projects
Selected production impact
HP · 2024 — Present · Public-safe summary
Enterprise systems. Measurable outcomes.
A public-safe view of engineering ownership across data platforms and Applied AI. Employer code, data and confidential architecture remain private.
0
1
Production GenAI engineering
Failur

In [63]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company/indivisual website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# # Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [58]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company/indivisual called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [64]:
get_brochure_user_prompt("Shivanand", "https://shivanandkumar.in")

Selecting relevant links for https://shivanandkumar.in by calling gpt-5-nano
Found 11 relevant links


"\nYou are looking at a company/indivisual called: Shivanand\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nShivanand Kumar | Data & Applied AI Engineer\n\nShivanand Kumar\nImpact\nProjects\nExperience\nContact\n01\nContext\nData · contracts · signals\n02\nRetrieval\nGrounded evidence\nAGENTIC\nWorkflow\n04\nEvaluation\nQuality · safety · traces\n05\nHuman approval\nControlled action\nBengaluru, India\nIIT Patna\nM.Tech · AI & Data Science Engineering\nData & Applied AI Engineer\nGenerative AI · Agentic AI · Databricks · Spark\nI build reliable data and AI systems—from Databricks and Spark platforms to GenAI automation, agentic workflows, retrieval and evaluation.\nSee production impact\nInspect public projects\nSelected production impact\nHP · 2024 — Present · Public-safe summary\nEnterprise systems. Measurable outcomes.\nA public-safe view of 

In [65]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [66]:
create_brochure("Shivanand", "https://shivanandkumar.in")

Selecting relevant links for https://shivanandkumar.in by calling gpt-5-nano
Found 7 relevant links


# Meet Shivanand Kumar  
*Your friendly neighborhood Data & Applied AI Engineer*

---

## Who is Shivanand?  
Imagine a wizard of data, conjuring spells on Databricks and Spark to tame the wildest AI beasts. That’s Shivanand Kumar—a master of Generative AI, Agentic AI, and data platform reliability, hailing from the halls of IIT Patna with an M.Tech in AI & Data Science Engineering.

Based in bustling Bengaluru, India, he crafts reliable, scalable, and downright impressive AI systems that don’t just talk the talk but walk the walk—delivering measurable outcomes for enterprises like HP and beyond.

---

## What’s Cooking in the AI Cauldron?  

- **Agentic AI workflows:** Turning complex data jobs into a symphony of automation.  
- **Databricks and Spark Knight:** Managing 40+ pipelines and slashing failure investigations by nearly 70%.  
- **AI-assisted Automation Supremo:** Automated contract generation cutting documentation effort from days to hours—because who has time for paperwork in 2024?  
- **Platform Reliability Guru:** Saved $1.55 million annually by deduplicating billions of data rows and reducing manual checks by up to 90%. Spoiler: no migration glitches here!  

Peek under the hood yourself—Shivanand’s work is open-source magic on GitHub. Excellence you can see and trust.

---

## Culture: Where AI Meets Human Touch  
Behind the code and pipelines is a philosophy: quality, safety, and human approval. Think of Shivanand’s systems as helpful robots that always check in with their human supervisor. Collaboration, transparency, and precision aren’t just buzzwords—they’re the secret sauce.

---

## Customers & Impact  
Big names, bigger results. From multinational HP to API-driven enterprise behemoths, Shivanand’s projects run heavy-duty data platforms with safety, speed, and savings that make CFOs smile.

---

## Careers: Join the Next-Gen AI Adventure  
Looking for a place where your data dreams can take flight? While Shivanand is currently flying solo, keep an eye out—joining forces with him means:

- Riding the leading edge of **Agentic AI** and **Generative AI** innovations.  
- Applying hands-on skills on real-world, impact-heavy enterprise projects.  
- Being part of an ecosystem inspired by the Indian Institute of Technology Patna—the birthplace of high-caliber AI talent.  

---

## Why IIT Patna? Because Brains Matter  
With a near 99% placement rate and a packed portfolio of 500+ research papers and 88+ patents, IIT Patna fuels talents like Shivanand. Their motto, "One who aspires wisdom, attains it," perfectly sums up the journey—from academic excellence to industry leadership.

---

## Contact & Connect  
Ready to harness the power of data and AI with Shivanand?  
Reach out, explore the open projects, or just drop a hello!  

- LinkedIn & GitHub links available upon request (just ping us!)  
- Based in Bengaluru, the Silicon Valley of India’s data future  

---

## Final Words: Data, AI, and a Dash of Wit  
Shivanand doesn’t just engineer systems—he engineers outcomes. If data is the new oil, consider him the refinery chief turning crude info into sparkling intelligence.  

Want reliable data flows, safer AI, and maybe a chuckle or two along the way? Shivanand’s your guy.  

---

*Go on, inspect the code, check the projects, and watch how AI magic happens.*  

**Shivanand Kumar — Building Future-Proof AI One Pipeline at a Time!**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [67]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [68]:
stream_brochure("Shivanand", "https://shivanandkumar.in")

Selecting relevant links for https://shivanandkumar.in by calling gpt-5-nano
Found 14 relevant links


# Meet Shivanand Kumar: Your Friendly Neighborhood Data & Applied AI Engineer

---

## Who is Shivanand?

If data and AI had a superhero, it would be Shivanand Kumar – wielding the powers of Databricks, Spark, and Generative AI to build reliable, scalable, and downright impressive AI systems. Based out of the tech-hub Bengaluru, India, and sharpened at the prestigious Indian Institute of Technology Patna (IIT Patna), Shivanand takes complex enterprise data challenges and turns them into measurable success stories.

---

## What Does Shivanand Do?

- **Generative AI & Agentic AI**: Automating workflows that think a little (or a lot) ahead so you don’t have to.
- **Data Products on Steroids**: Crafting contracts, schemas, and transformations across 47+ Databricks pipelines – making days’ work happen in hours.
- **Failure Analysis Jedi**: Slashing 70% of effort investigating data failures by orchestrating over 40 Databricks jobs into a smooth-running machine.
- **Platform Reliability Guru**: Reducing recurring manual checks by up to 90%, saving millions annually, and shrinking PySpark workloads with ninja-like efficiency.

---

## The Impact

- **$1.55M in Annual Savings**
- **85-90% Reduction** in manual data platform checks
- **75% Runtime Slashing** on heavy PySpark jobs
- **20+ Transformations** automated in complex governed workflows
- **Zero Downtime Migration** of 40 Databricks pipelines — because downtime is so last decade

---

## Why Work with Shivanand? The Culture & Mindset

- **Precision with Playfulness**: Serious about engineering quality but knows that good humor fuels creativity.
- **Transparency Matters**: Openly shares public, reproducible projects so you trust not just the claims but the code behind them.
- **Human-In-The-Loop**: AI’s best friend is human approval – because machine smarts need human heartbeats.
- **Continuous Learner & Innovator**: Always exploring cutting-edge AI while driving impact in real-world enterprise environments.

---

## Shivanand's Tribe: Customers & Collaborators

Teaming up with enterprise giants like **HP**, tackling production-scale AI challenges that demand security, reliability, and measurable outcomes. If your company needs a data and AI engineer who speaks Slack, SQL, and subtle sarcasm, look no further.

---

## Join the Journey

Dreaming of a career where your AI magic touches millions and your code matter? Shivanand’s story inspires a career in:

- Data Engineering & Analytics
- Applied AI and Automation
- Scalable Cloud and Spark Ecosystems
- Workflow Orchestration and System Reliability

Learn from the best, contribute boldly, and build the future where data and AI work hand-in-hand with humans.

---

## Explore More from Shivanand

- **Inspect the magic yourself:** Public reproducible projects await curious minds at GitHub — omni-shiva
- **Academic Roots:** IIT Patna, where “One who aspires wisdom, attains it” is not just a motto but a way of life.
- **Connect:** Whether you want to talk AI, data, or just share a good tech pun, Shivanand’s contact is just a click away.

---

## Final Word: Because Data Deserves a Hero

In a world drowning in data, Shivanand Kumar is the engineer you want on your team: part magician, part scientist, and all heart. Expect robust AI systems, a sprinkle of brilliance, and just enough wit to keep the servers and spirits up.

---

*Be bold. Be data-driven. Be Shivanand-inspired.*

---

*Contact & Projects: [GitHub omni-shiva] | [LinkedIn Profile]*  

*Based in Bengaluru, engineered in India, impacting enterprises globally.*

In [69]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Shivanand", "https://shivanandkumar.in")

Selecting relevant links for https://shivanandkumar.in by calling gpt-5-nano
Found 7 relevant links


# Meet Shivanand Kumar: The Data & Applied AI Dynamo 🚀

---

## Who is Shivanand?

Imagine a wizard who crafts magic not with wands, but with **Databricks, Spark, and Generative AI** — that’s Shivanand Kumar for you! Based in Bengaluru, India, and a proud alumni of IIT Patna’s revered AI & Data Science M.Tech program, Shivanand is the master of building *reliable, scalable,* and *human-approved* AI systems that power enterprises like HP with measurable results.

---

## What Does Shivanand Actually Do?

- **Production GenAI Engineering:**  
  Conjured a graph-orchestrated workflow running 40 Databricks jobs to crush failure investigations by a whopping **70%!** Imagine slashing tedious tasks so much your coffee breaks quadruple. ☕

- **AI-Assisted Automation:**  
  Automated 47 data pipelines, turning documentation days into mere hours and making validation a breeze. Yes, AI approved the AI’s work—because why not?

- **Platform Reliability Hero:**  
  Enhanced monitoring and efficiency across 54 data pipelines with results like:  
  - **85–90% fewer manual checks** (humans get fewer headaches)  
  - Deduplication slashed from **21 billion to 400 million rows** (because who wants redundant data?)  
  - Saved a cool **$1.55 million annually** (enough to fund a robot uprising... or just coffee)

---

## The Culture at Shivanand World

- **Innovation Meets Pragmatism:** No pie-in-the-sky here; every AI and data workflow is battle-tested to boost real-world business outcomes.  
- **Transparency is King:** Public-safe summaries and reproducible projects mean Shivanand’s work plays nice with anyone who wants to peek behind the curtain.  
- **Learning & Excellence:** With roots at IIT Patna—an institution where *“One who aspires wisdom, attains it”* is the mantra—expect a culture of relentless curiosity and academic rigor.

---

## Why Partner or Work With Shivanand?

- **For Customers:**  
  If enterprise-grade reliability, cutting-edge AI solutions, and cost-saving data systems excite you, Shivanand's your go-to engineer. Your operational hiccups will be fewer, your workflows smarter, and your ROI shinier.

- **For Investors:**  
  Investing in Shivanand is like betting on the future of AI-powered enterprise efficiency. With an arsenal of patented knowledge (courtesy of IIT Patna’s research ecosystem) and proven savings in millions, expect steady tech-driven growth.

- **For Recruits:**  
  Fancy working with an AI/Data guru who blends rigorous science with down-to-earth engineering? Dive into advanced AI automation, explore generative models, and help build systems that actually *avoid* breaking (a rare talent indeed). Bonus: coffee breaks might get longer with those efficiency gains.

---

## Bonus: Education Roots

Shivanand’s alma mater, **IIT Patna**, isn’t just any institute — it’s a top 10 national ranked powerhouse with a placement rate north of 98.5%, 500+ research papers, interdisciplinary supercomputing centers, and a vibrant startup culture. So yes, the foundation is solid.

---

## Want to See the Magic?

Shivanand's projects are **open to public inspection** with independent builds on GitHub for the curious and the skeptical alike. Because when you’re confident about your work, you put it out there for everyone to admire.

---

# Ready to unleash data & AI brilliance with Shivanand?

Reach out, collaborate or join the futuristic journey of turning data chaos into reliable, agentic intelligence!

---

*Shivanand Kumar* — Making AI not just smart, but sensible.  
*Bengaluru, India | M.Tech @ IIT Patna | Data & Applied AI Engineer Extraordinaire*

---

*Disclaimer: No AI systems were harmed in the making of these pipelines.* 🤖✨

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>